In [1]:
import pandas as pd
import json
from ydata_profiling import ProfileReport

In [2]:
# Načtení datasetu
df = pd.read_csv("../data/AirQualityUCI.csv", sep=';', decimal=',')

# Odstranění prázdných sloupců
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Odstranění prázdných řádků
df.dropna(how='all', inplace=True)

df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


In [3]:
# Vytvoření profiling reportu
profile = ProfileReport(df, title="Air Quality Profiling", explorative=True)

In [4]:
# Extrakce profilu jako slovníku pro další porovnání
desc = profile.get_description()
profile_dict = desc.variables

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:00<00:00, 74.47it/s]


In [5]:
# Extrakce ze slovníku
schema = {}

for var_name, var_info in profile_dict.items():
    if not isinstance(var_info, dict):
        continue

    if "count" not in var_info or "min" not in var_info or "max" not in var_info:
        continue

    # Získání četností s hodnotami
    value_counts_raw = var_info.get("value_counts_without_nan", [])
    value_counts = []

    if isinstance(value_counts_raw, pd.Series):
        for val, count in value_counts_raw.items():
            value_counts.append({"value": str(val), "count": int(count)})
    elif isinstance(value_counts_raw, list):  # pokud už je to list dictů
        value_counts = value_counts_raw[:10]
    else:
        value_counts = []

    # Zápis do schématu
    schema[var_name] = {
        "count": var_info.get("count"),
        "min": str(var_info.get("min")),
        "max": str(var_info.get("max")),
        "value_counts": value_counts[:10]  # jen top 10
    }

In [6]:
# Uložení jako JSON
with open("../missing/export_encoded/air_quality_profile.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

In [7]:
# Načtení JSON dat
with open("../missing/export_encoded/air_quality_profile.json", "r", encoding="utf-8") as f:
    profile_data = json.load(f)

In [8]:
# Slovník pro podezřelé hodnoty
suspect_values = {}

# Projdeme každou proměnnou v profilu
for column_name, info in schema.items():
    total_count = info["count"]
    min_value = info["min"]
    max_value = info["max"]
    top_values = info["value_counts"]

    for entry in top_values:
        value = entry["value"]
        count = entry["count"]
        rel_freq = count / total_count if total_count > 0 else 0

        # Podezřelá hodnota = často se vyskytuje nebo je extrémní (min nebo max)
        if rel_freq >= 0.01 or value == min_value or value == max_value:
            if value not in suspect_values:
                suspect_values[value] = {"columns": [], "total_occurrences": 0}
            suspect_values[value]["columns"].append(column_name)
            suspect_values[value]["total_occurrences"] += count

# Pouze hodnoty, které se objevují napříč více sloupci
filtered_suspects = {
    value: info
    for value, info in suspect_values.items()
    if len(set(info["columns"])) > 1
}

In [9]:
# Uložení jako JSON
with open("../missing/export_encoded/suspects.json", "w", encoding="utf-8") as f:
    json.dump(filtered_suspects, f, ensure_ascii=False, indent=2)